In [31]:
import cv2
import mediapipe as mp
import numpy as np

In [32]:
import cv2

cap = cv2.VideoCapture(0)
print(cap.isOpened())

True


In [ ]:
#Integrantes
#97803 - Samuel Mota Gama de Abreu
#552041 - Gabriel Valério Gouveia#
#551351 - Thiago Ratão Passerini

In [ ]:
#Como rodar o projeto (passo a passo simples)
#Instale o Python 3.10 (essa versão é necessária por causa do MediaPipe).
#Instale as bibliotecas:
#pip install opencv-python mediapipe==0.10.9 numpy
#Abra o arquivo no Jupyter Notebook (.ipynb) e verifique se o kernel está usando o Python 3.10.
#Se algo não funcionar:
#Reinicie o kernel
#Verifique se as bibliotecas foram instaladas corretamente
#este se a webcam está funcionando
#O que deve acontecer

#Vai abrir uma tela com a sua câmera, mostrando seu corpo sendo rastreado, contando as repetições e dando feedback em tempo real.

In [33]:
#Descrição do Projeto e Divisão de Responsabilidades

#Este projeto consiste em um sistema de contagem de exercícios físicos em tempo real utilizando visão computacional com MediaPipe e OpenCV.
#A aplicação captura a imagem da webcam, identifica pontos do corpo e calcula ângulos para reconhecer movimentos como agachamento, flexão e polichinelo,
#além de fornecer feedback visual e acompanhamento da execução.

#Samuel foi responsável pela implementação da lógica principal do sistema, incluindo a detecção dos movimentos, cálculo dos ângulos e contagem das repetições. 
#Gabriel atuou no desenvolvimento da interface visual (HUD), exibindo informações como progresso, fase do exercício e feedback ao usuário. 
#Thiago contribuiu na integração das tecnologias, testes do sistema e ajustes de desempenho e usabilidade, garantindo o funcionamento em tempo real.

#O projeto tem como objetivo oferecer uma solução acessível para auxiliar na prática de exercícios físicos, 
#combinando precisão na detecção com uma interface intuitiva para melhorar a experiência do usuário.

In [34]:
def calcular_angulo(a, b, c):
    """Calcula o ângulo (graus) formado pelos pontos a-b-c, onde b é o vértice."""
    a, b, c = np.array(a), np.array(b), np.array(c)
    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - \
              np.arctan2(a[1] - b[1], a[0] - b[0])
    angulo = np.abs(np.degrees(radians))
    if angulo > 180:
        angulo = 360 - angulo
    return angulo
 
 
def ponto(landmarks, idx, w, h):
    """Retorna as coordenadas (x, y) em pixels de um landmark."""
    lm = landmarks[idx]
    return [int(lm.x * w), int(lm.y * h)]

In [35]:
# ──────────────────────────────────────────────
# Lógica de detecção por exercício
# ──────────────────────────────────────────────
 
class ExercicioBase:
    """Classe base para todos os exercícios."""
 
    NOME = "Exercicio"
    COR  = (255, 255, 255)      # BGR
 
    def __init__(self):
        self.contador  = 0
        self.fase      = "inicio"   # estado interno da máquina de fases
        self.feedback  = ""         # mensagem de postura ao usuário
        self.progresso = 0.0        # 0.0 → 1.0 para a barra de progresso
 
    def resetar(self):
        self.contador  = 0
        self.fase      = "inicio"
        self.feedback  = ""
        self.progresso = 0.0
 
    def processar(self, landmarks, w, h):
        """Deve ser sobrescrito; atualiza contador, fase, feedback e progresso."""
        raise NotImplementedError

In [36]:
# ── Agachamento ──────────────────────────────
class Agachamento(ExercicioBase):
    NOME = "Agachamento"
    COR  = (0, 200, 255)   # laranja
 
    # Limiares de ângulo no joelho
    ANGULO_BAIXO  = 90    # posição agachada
    ANGULO_CIMA   = 160   # posição em pé
 
    def processar(self, landmarks, w, h):
        # Usa o lado mais visível (quadril–joelho–tornozelo)
        for prefixo in (
            (mp.solutions.pose.PoseLandmark.LEFT_HIP,
             mp.solutions.pose.PoseLandmark.LEFT_KNEE,
             mp.solutions.pose.PoseLandmark.LEFT_ANKLE),
            (mp.solutions.pose.PoseLandmark.RIGHT_HIP,
             mp.solutions.pose.PoseLandmark.RIGHT_KNEE,
             mp.solutions.pose.PoseLandmark.RIGHT_ANKLE),
        ):
            hip, knee, ankle = prefixo
            if landmarks[knee.value].visibility < 0.5:
                continue
 
            a = ponto(landmarks, hip.value,   w, h)
            b = ponto(landmarks, knee.value,  w, h)
            c = ponto(landmarks, ankle.value, w, h)
            ang = calcular_angulo(a, b, c)
 
            # Progresso visual: 1.0 = em pé, 0.0 = agachado
            self.progresso = np.clip(
                (ang - self.ANGULO_BAIXO) / (self.ANGULO_CIMA - self.ANGULO_BAIXO), 0, 1
            )
 
            if ang > self.ANGULO_CIMA:
                if self.fase == "baixo":
                    self.contador += 1
                self.fase = "cima"
                self.feedback = "Em pe ✓"
            elif ang < self.ANGULO_BAIXO:
                self.fase = "baixo"
                self.feedback = "Agachado ✓"
            else:
                if self.fase == "cima":
                    self.feedback = "Desca mais!"
                else:
                    self.feedback = "Suba mais!"
            break

In [37]:
# ── Flexão ───────────────────────────────────
class Flexao(ExercicioBase):
    NOME = "Flexao"
    COR  = (0, 255, 128)   # verde
 
    ANGULO_BAIXO = 80    # cotovelo dobrado
    ANGULO_CIMA  = 155   # braço estendido
 
    def processar(self, landmarks, w, h):
        for prefixo in (
            (mp.solutions.pose.PoseLandmark.LEFT_SHOULDER,
             mp.solutions.pose.PoseLandmark.LEFT_ELBOW,
             mp.solutions.pose.PoseLandmark.LEFT_WRIST),
            (mp.solutions.pose.PoseLandmark.RIGHT_SHOULDER,
             mp.solutions.pose.PoseLandmark.RIGHT_ELBOW,
             mp.solutions.pose.PoseLandmark.RIGHT_WRIST),
        ):
            shoulder, elbow, wrist = prefixo
            if landmarks[elbow.value].visibility < 0.5:
                continue
 
            a = ponto(landmarks, shoulder.value, w, h)
            b = ponto(landmarks, elbow.value,    w, h)
            c = ponto(landmarks, wrist.value,    w, h)
            ang = calcular_angulo(a, b, c)
 
            self.progresso = np.clip(
                (ang - self.ANGULO_BAIXO) / (self.ANGULO_CIMA - self.ANGULO_BAIXO), 0, 1
            )
 
            if ang > self.ANGULO_CIMA:
                if self.fase == "baixo":
                    self.contador += 1
                self.fase = "cima"
                self.feedback = "Braço estendido ✓"
            elif ang < self.ANGULO_BAIXO:
                self.fase = "baixo"
                self.feedback = "Cotovelo dobrado ✓"
            else:
                if self.fase == "cima":
                    self.feedback = "Desca mais!"
                else:
                    self.feedback = "Suba mais!"
            break

In [38]:
# ── Polichinelo ──────────────────────────────
 
class Polichinelo(ExercicioBase):
    NOME = "Polichinelo"
    COR  = (255, 80, 200)   # rosa
 
    # Usa a distância relativa entre os pulsos para detectar abertura/fechamento
    DIST_ABERTO  = 0.45   # fração da largura do frame
    DIST_FECHADO = 0.15
 
    def processar(self, landmarks, w, h):
        PL = mp.solutions.pose.PoseLandmark
        lm = landmarks
 
        # Visibilidade dos pulsos
        if lm[PL.LEFT_WRIST.value].visibility < 0.4 or \
           lm[PL.RIGHT_WRIST.value].visibility < 0.4:
            self.feedback = "Fique de frente para a camera"
            return
 
        pw = ponto(lm, PL.LEFT_WRIST.value,  w, h)
        qw = ponto(lm, PL.RIGHT_WRIST.value, w, h)
 
        dist = abs(pw[0] - qw[0]) / w   # distância horizontal normalizada
 
        self.progresso = np.clip(
            (dist - self.DIST_FECHADO) / (self.DIST_ABERTO - self.DIST_FECHADO), 0, 1
        )
 
        if dist > self.DIST_ABERTO:
            if self.fase == "fechado":
                self.contador += 1
            self.fase = "aberto"
            self.feedback = "Aberto ✓"
        elif dist < self.DIST_FECHADO:
            self.fase = "fechado"
            self.feedback = "Fechado ✓"
        else:
            self.feedback = "Continue o movimento!"

In [39]:
# ──────────────────────────────────────────────
# Renderização de HUD
# ──────────────────────────────────────────────
 
FONTE       = cv2.FONT_HERSHEY_DUPLEX
FONTE_MONO  = cv2.FONT_HERSHEY_SIMPLEX
PRETO       = (0,   0,   0)
BRANCO      = (255, 255, 255)
CINZA_ESC   = (30,  30,  30)
CINZA_MEIO  = (80,  80,  80)
VERMELHO    = (60,  60, 220)
VERDE       = (60, 200,  60)
 
 
def retangulo_arredondado(img, pt1, pt2, cor, raio=12, espessura=-1, alpha=1.0):
    """Desenha um retângulo com cantos arredondados."""
    overlay = img.copy()
    x1, y1 = pt1
    x2, y2 = pt2
    cv2.rectangle(overlay, (x1 + raio, y1), (x2 - raio, y2), cor, espessura)
    cv2.rectangle(overlay, (x1, y1 + raio), (x2, y2 - raio), cor, espessura)
    for cx, cy in [(x1+raio, y1+raio), (x2-raio, y1+raio),
                   (x1+raio, y2-raio), (x2-raio, y2-raio)]:
        cv2.circle(overlay, (cx, cy), raio, cor, espessura)
    if alpha < 1.0:
        cv2.addWeighted(overlay, alpha, img, 1 - alpha, 0, img)
    else:
        img[:] = overlay
 
 
def texto_centralizado(img, texto, cx, cy, escala, cor, espessura=1):
    (tw, th), _ = cv2.getTextSize(texto, FONTE, escala, espessura)
    cv2.putText(img, texto, (cx - tw // 2, cy + th // 2),
                FONTE, escala, cor, espessura, cv2.LINE_AA)
 
 
def desenhar_barra_progresso(img, x, y, largura, altura, progresso, cor_barra):
    """Barra vertical de progresso (cresce para cima)."""
    # Fundo
    retangulo_arredondado(img, (x, y), (x + largura, y + altura),
                          CINZA_ESC, raio=6)
    # Preenchimento
    fill_h = int(altura * progresso)
    if fill_h > 4:
        retangulo_arredondado(img, (x, y + altura - fill_h),
                              (x + largura, y + altura),
                              cor_barra, raio=6)
    # Borda
    cv2.rectangle(img, (x, y), (x + largura, y + altura), CINZA_MEIO, 1)
 
 
def desenhar_hud(frame, exercicio, exercicios, idx_atual):
    h, w = frame.shape[:2]
 
    # ── Painel lateral esquerdo ──────────────────
    painel_w = 220
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (painel_w, h), (15, 15, 15), -1)
    cv2.addWeighted(overlay, 0.75, frame, 0.25, 0, frame)
 
    # Logo / título
    cv2.putText(frame, "EXERCISE", (12, 32), FONTE_MONO, 0.65, exercicio.COR, 1, cv2.LINE_AA)
    cv2.putText(frame, "COUNTER",  (12, 54), FONTE_MONO, 0.65, exercicio.COR, 1, cv2.LINE_AA)
    cv2.line(frame, (12, 62), (painel_w - 12, 62), CINZA_MEIO, 1)
 
    # Contador grande
    cv2.putText(frame, "REPS", (12, 88), FONTE_MONO, 0.45, CINZA_MEIO, 1, cv2.LINE_AA)
    texto_centralizado(frame, str(exercicio.contador),
                       painel_w // 2, 145, 3.5, exercicio.COR, 4)
 
    # Barra de progresso
    bx, by, bw, bh = 84, 175, 52, 130
    desenhar_barra_progresso(frame, bx, by, bw, bh, exercicio.progresso, exercicio.COR)
    cv2.putText(frame, f"{int(exercicio.progresso * 100)}%",
                (bx, by + bh + 18), FONTE_MONO, 0.45, CINZA_MEIO, 1, cv2.LINE_AA)
 
    # Fase atual
    fase_str = exercicio.fase.upper()
    cv2.putText(frame, "FASE", (12, 325), FONTE_MONO, 0.4, CINZA_MEIO, 1, cv2.LINE_AA)
    cv2.putText(frame, fase_str, (12, 348), FONTE_MONO, 0.55, BRANCO, 1, cv2.LINE_AA)
 
    # Exercício ativo
    cv2.putText(frame, "EXERCICIO", (12, 385), FONTE_MONO, 0.4, CINZA_MEIO, 1, cv2.LINE_AA)
    cv2.putText(frame, exercicio.NOME, (12, 408), FONTE_MONO, 0.52, exercicio.COR, 1, cv2.LINE_AA)
 
    # ── Feedback de postura (topo central) ───────
    if exercicio.feedback:
        msg = exercicio.feedback
        (tw, _), _ = cv2.getTextSize(msg, FONTE_MONO, 0.7, 2)
        fx = painel_w + (w - painel_w - tw) // 2
        fy = 36
        # Sombra
        cv2.putText(frame, msg, (fx + 2, fy + 2), FONTE_MONO, 0.7, PRETO,   2, cv2.LINE_AA)
        cv2.putText(frame, msg, (fx,     fy),      FONTE_MONO, 0.7, exercicio.COR, 2, cv2.LINE_AA)
 
    # ── Mini-menu de exercícios (rodapé) ─────────
    menu_h = 40
    cv2.rectangle(frame, (painel_w, h - menu_h), (w, h), (20, 20, 20), -1)
    opcoes = [(i + 1, ex.NOME, ex.COR) for i, ex in enumerate(exercicios)]
    slot_w = (w - painel_w) // len(opcoes)
    for i, (num, nome, cor) in enumerate(opcoes):
        sx = painel_w + i * slot_w
        ativo = (i == idx_atual)
        cor_texto = cor if ativo else CINZA_MEIO
        borda = 2 if ativo else 0
        if ativo:
            cv2.rectangle(frame, (sx, h - menu_h), (sx + slot_w, h),
                          (40, 40, 40), -1)
        cv2.putText(frame, f"[{num}] {nome}",
                    (sx + 10, h - 13), FONTE_MONO, 0.45, cor_texto, borda if borda else 1, cv2.LINE_AA)
 
    # ── Dica de teclas (canto inferior direito) ──
    dica = "[r] Reset   [q] Sair"
    (dw, _), _ = cv2.getTextSize(dica, FONTE_MONO, 0.38, 1)
    cv2.putText(frame, dica, (w - dw - 10, h - menu_h - 8),
                FONTE_MONO, 0.38, CINZA_MEIO, 1, cv2.LINE_AA)
 
    # Linha separadora do painel
    cv2.line(frame, (painel_w, 0), (painel_w, h), CINZA_MEIO, 1)

In [40]:
def main():
    exercicios = [Agachamento(), Flexao(), Polichinelo()]
    idx = 0   # exercício ativo
 
    mp_pose     = mp.solutions.pose
    mp_drawing  = mp.solutions.drawing_utils
    mp_styles   = mp.solutions.drawing_styles
 
    pose = mp_pose.Pose(
        model_complexity=1,
        smooth_landmarks=True,
        min_detection_confidence=0.6,
        min_tracking_confidence=0.6,
    )
 
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("Erro: nao foi possivel abrir a camera.")
        return
 
    # Tenta 720p; se não suportar, aceita o padrão
    cap.set(cv2.CAP_PROP_FRAME_WIDTH,  1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
 
    print("="*52)
    print("  Contador de Exercicios com Feedback de Postura")
    print("="*52)
    print("  [1] Agachamento  [2] Flexao  [3] Polichinelo")
    print("  [r] Resetar contador         [q] Sair")
    print("="*52)
 
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
 
        frame = cv2.flip(frame, 1)   # espelho para ser intuitivo
        h, w = frame.shape[:2]
 
        # ── Inferência MediaPipe ─────────────────
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        rgb.flags.writeable = False
        results = pose.process(rgb)
        rgb.flags.writeable = True
 
        # ── Skeleton ────────────────────────────
        if results.pose_landmarks:
            mp_drawing.draw_landmarks(
                frame,
                results.pose_landmarks,
                mp_pose.POSE_CONNECTIONS,
                landmark_drawing_spec=mp_drawing.DrawingSpec(
                    color=(200, 200, 200), thickness=2, circle_radius=3),
                connection_drawing_spec=mp_drawing.DrawingSpec(
                    color=(100, 100, 100), thickness=2),
            )
 
            lm = results.pose_landmarks.landmark
            exercicios[idx].processar(lm, w, h)
        else:
            exercicios[idx].feedback = "Posicione-se na camera"
 
        # ── HUD ──────────────────────────────────
        desenhar_hud(frame, exercicios[idx], exercicios, idx)
 
        cv2.imshow("Exercise Counter", frame)
 
        # ── Teclas ───────────────────────────────
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            break
        elif key == ord('r'):
            exercicios[idx].resetar()
            print(f"Contador de {exercicios[idx].NOME} resetado.")
        elif key == ord('1'):
            idx = 0
            print(f"Exercicio: {exercicios[idx].NOME}")
        elif key == ord('2'):
            idx = 1
            print(f"Exercicio: {exercicios[idx].NOME}")
        elif key == ord('3'):
            idx = 2
            print(f"Exercicio: {exercicios[idx].NOME}")
 
    cap.release()
    pose.close()
    cv2.destroyAllWindows()
    print("\nResumo final:")
    for ex in exercicios:
        print(f"  {ex.NOME}: {ex.contador} repeticoes")
 
 
if __name__ == "__main__":
    main()

  Contador de Exercicios com Feedback de Postura
  [1] Agachamento  [2] Flexao  [3] Polichinelo
  [r] Resetar contador         [q] Sair
Exercicio: Flexao
Exercicio: Polichinelo
Exercicio: Agachamento
Contador de Agachamento resetado.
Exercicio: Flexao
Contador de Flexao resetado.
Contador de Flexao resetado.
Contador de Flexao resetado.
Exercicio: Agachamento
Contador de Agachamento resetado.
Exercicio: Polichinelo
Contador de Polichinelo resetado.

Resumo final:
  Agachamento: 3 repeticoes
  Flexao: 6 repeticoes
  Polichinelo: 8 repeticoes
